# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [3]:
import gradio as gr
from dotenv import load_dotenv
from mistralai.client import Mistral
import sqlite3
import os
import base64
from io import BytesIO

In [4]:
# So what I would like to do, is to generate a tutor to learn me and my wife to speak French. 
    # Requirements
    # I want to run it on my local network
    # A user should be able to speak to it, and it should reply back and correct the user in case the user made an error.
    # It should remember the conversation history
    # The conversations should be relevant to the user. So topics can vary. I could use a tool to find the relevant topics.

In [5]:
load_dotenv(override=True)
mistral_api_key = os.getenv('MISTRAL_API_KEY')

In [6]:
if mistral_api_key:
    print(f'Mistral API key found and begins with {mistral_api_key[0:5]}')

Mistral API key found and begins with qsF5g


In [7]:
MODEL = "mistral-small-2603"
Mistral= Mistral(api_key=mistral_api_key)

In [8]:
#Some parameters:
Conv_model = 'mistrall-small-latest'
Voice_model = 'voxtral-mini-tts-2603'
Voice_id = 'e90ce106-8ce1-401f-ab90-cde11dbaca80'

system_message = "Je bent een Franse tutor. Je helpt om Nikki en Thijs Frans te leren. Het idee is dat je feilloos tussen Nederlands en Frans kan switchen en een conversatie " \
"Frans aangaat. Wanneer Nikki of Thijs een fout maakt, voel je vrij de fout te corrigeren." \
"Start de conversatie eerst met het stellen van de vraag of je met Thijs of met Nikki spreekt. Een paar relevante onderwerpen voor Thijs (A12 / B1 niveau):" \
"Hij wilt graag in Frankrijk gaan werken en moet business frans leren. Geinteresseerd in data science, techniek, wetenschap." \
"Nikki (A1 niveau) wilt een b&b openen in Frankrijk, houdt van dierentuinen en de natuur in Frankrijk. Ze vind het leuk om feitjes te leren over de Dordogne (perigord) om zo " \
"de streek beter te leren kennen. In het geval je meer interesses wilt weten kan je ook de intresse tool gebruiken."

In [9]:
DB = "interest.db"

In [10]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS user_interests (name TEXT PRIMARY KEY, interest TEXT)')
    conn.commit()

In [11]:
def add_interests():
    name = input("Please state your name:")
    interests = input(f'Hello {name}! Can you state me your interests?')
    table = "user_interests"

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        QUERY = f"INSERT INTO {table} (name , interest) VALUES (?,?) ON CONFLICT(name) DO UPDATE SET interest = COALESCE(interest, '') || ', ' || EXCLUDED.interest"
        cursor.execute(QUERY,(name,interests))
    return interests.split(" ")

In [12]:
add_interests()

['']

In [13]:
def get_user_interests(name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT interest FROM user_interests WHERE name = ?', (name,))
        results = cursor.fetchone()
        return results
        

In [14]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("DELETE FROM user_interests WHERE name = ?",('Thijs',))

In [15]:
get_user_interests("Thijs")

In [ ]:
def talker(message):
    response = Mistral.audio.speech.complete(
      model="voxtral-mini-tts-2603",
      voice_id="e90ce106-8ce1-401f-ab90-cde11dbaca80",
      input= message,
      response_format='mp3'
    )

    return base64.b64decode(response.audio_data)

In [27]:
# I got an error if the response was too big. 
def talker(message, max_chunk=500):
    # Split long messages into chunks
    chunks = [message[i:i+max_chunk] for i in range(0, len(message), max_chunk)]
    audio_parts = []

    for chunk in chunks:
        for attempt in range(3):  # Retry up to 3 times
            try:
                response = Mistral.audio.speech.complete(
                    model="voxtral-mini-tts-2603",
                    voice_id="e90ce106-8ce1-401f-ab90-cde11dbaca80",
                    input=chunk,
                    response_format='mp3'
                )
                audio_parts.append(base64.b64decode(response.audio_data))
                break
            except Exception as e:
                if attempt == 2:
                    raise RuntimeError(f"Failed to generate TTS for chunk: {chunk[:50]}...") from e
                continue

    return b''.join(audio_parts)  # Combine all chunks

In [28]:
def conversational_tutor(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = Mistral.chat.complete(model=MODEL, messages=messages)



    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    return history, voice

In [29]:
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo", device='cuda' # of "whisper-large-v3-turbo" (sneller)
)

Device set to use cuda


In [30]:
def asr(audio_filepath):
    with open(audio_filepath, 'rb') as f:
        audio_bytes = f.read()
    result = Mistral.audio.transcriptions.complete(
        model='voxtral-mini-latest',
        file={
            "content": audio_bytes, 
            'file_name': 'audio.mp3'
        }
    )
    return result.text 

In [32]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

def process_audio(audio_file, history):
    if audio_file is None:
        return history, None
    
    text = asr(audio_file)  # Now returns text, not the full response
    history.append({"role": "user", "content": text})
    history, voice = conversational_tutor(history)
    return history, voice

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
         audio_input = gr.Audio(type='filepath', sources=['microphone'])
    
  # Automatisch: audio-input → ASR → chat → TTS
    audio_input.change(
        process_audio,
        inputs=[audio_input, chatbot],
        outputs=[chatbot, audio_output],
    )

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        conversational_tutor, inputs=chatbot, outputs=[chatbot, audio_output]
    )


ui.launch(auth=("Thijs", "frenchtutor"), share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://d8f920b5d5986e932a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
